# Pipeline 3: Social Media Donation Conversion Model

## 1. Problem Framing

### Business Problem
Social media is the organization's primary channel for reaching potential donors, but the founders freely admit they "are not experienced with social media." They post sporadically and struggle with fundamental questions: What should they post? On which platforms? How often? What time of day? What content actually leads to donations versus just generating likes?

Without a marketing team and without budget to hire one, the organization needs data-driven guidance on how to make every post count. The difference between a well-crafted post and a generic one could mean thousands of pesos in donations — money that directly funds meals, education, and counseling for the girls.

### Who Cares
- **Founders / leadership**: Need to understand which social media activities actually translate to donations, not just engagement metrics.
- **Staff managing social media**: Need actionable guidance on content strategy — what to post, when, and where.
- **Donors and potential donors**: Benefit from more compelling, well-timed content that connects them to the organization's mission.

### Why It Matters
Social media is free. The organization can't buy TV ads or hire PR firms. Every post is an opportunity to reach donors, but only if the content is strategic. A model that connects post characteristics to donation outcomes turns guesswork into a data-informed content strategy.

### Approach: Predictive AND Explanatory
- **Predictive goal**: Build a regression model that predicts the estimated donation value (PHP) a post will generate based on its characteristics. This could power an interactive "content advisor" tool in the web app.
- **Explanatory goal**: Build an OLS regression with interpretable coefficients to answer specific strategic questions: Do calls to action actually drive donations? Are resident stories more effective than generic content? Does boosting pay off? Which platforms convert best? These coefficients become the foundation of a content strategy playbook.

## 2. Data Acquisition, Preparation & Exploration

The primary data source is `social_media_posts` (812 posts across 7 platforms). We also join to `donations` via `referral_post_id` for ground-truth attribution analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 6)

# ----- Load data -----
DATA_DIR = "../lighthouse_csv_v7"  # Adjust this path relative to your notebook location

social = pd.read_csv(f"{DATA_DIR}/social_media_posts.csv")
donations = pd.read_csv(f"{DATA_DIR}/donations.csv")

print(f"Social media posts: {social.shape}")
print(f"Donations: {donations.shape}")
print(f"\nPlatforms: {social['platform'].nunique()}")
print(f"Date range: {social['created_at'].min()} to {social['created_at'].max()}")

In [ ]:
# Basic feature engineering
social['created_at'] = pd.to_datetime(social['created_at'])
social['is_story'] = social['features_resident_story'].astype(int)
social['is_cta'] = social['has_call_to_action'].astype(int)
social['is_boosted_flag'] = social['is_boosted'].astype(int)
social['engagements'] = social[['likes', 'comments', 'shares']].fillna(0).sum(axis=1)

# Log-transform the target (highly right-skewed donation values)
social['log_donation_value'] = np.log1p(social['estimated_donation_value_php'])

# Weekend flag
social['is_weekend'] = social['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

# Time of day buckets
social['time_bucket'] = pd.cut(social['post_hour'], bins=[0, 6, 12, 18, 24], 
                                labels=['Night', 'Morning', 'Afternoon', 'Evening'], right=False)

print(f"Target variable stats (estimated_donation_value_php):")
print(social['estimated_donation_value_php'].describe().round(2))
print(f"\nPosts with zero donation value: {(social['estimated_donation_value_php'] == 0).sum()} ({(social['estimated_donation_value_php'] == 0).mean():.1%})")

In [ ]:
# Outlier detection and handling
# Check for extreme outliers in the target variable
Q1 = social['estimated_donation_value_php'].quantile(0.25)
Q3 = social['estimated_donation_value_php'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = social[(social['estimated_donation_value_php'] < lower_bound) | 
                  (social['estimated_donation_value_php'] > upper_bound)]
print(f'Outlier Analysis (IQR method):')
print(f'  IQR: {IQR:,.0f}, Lower: {lower_bound:,.0f}, Upper: {upper_bound:,.0f}')
print(f'  Outliers found: {len(outliers)} ({len(outliers)/len(social):.1%})')
print(f'  Max value: {social["estimated_donation_value_php"].max():,.0f}')

# We use log transformation (log1p) rather than removing outliers,
# because extreme donation values are real and meaningful — they represent
# viral posts or major donor conversions. The log transform compresses the
# scale while preserving the information.
print('\nStrategy: Using log1p transformation to handle right-skew rather than removing outliers.')
print('This preserves valuable signal from high-performing posts.')

# Check numeric features for outliers
print('\nNumeric feature outlier summary (values beyond 3 std from mean):')
for col in ['caption_length', 'num_hashtags', 'mentions_count', 'follower_count_at_post']:
    mean, std = social[col].mean(), social[col].std()
    n_outliers = ((social[col] < mean - 3*std) | (social[col] > mean + 3*std)).sum()
    print(f'  {col}: {n_outliers} outliers')


### Exploratory Analysis

In [ ]:
# Platform comparison
platform_perf = social.groupby('platform').agg(
    posts=('post_id', 'count'),
    avg_engagement_rate=('engagement_rate', 'mean'),
    avg_click_throughs=('click_throughs', 'mean'),
    total_referrals=('donation_referrals', 'sum'),
    total_donation_value=('estimated_donation_value_php', 'sum'),
    avg_donation_value=('estimated_donation_value_php', 'mean')
).round(2).sort_values('avg_donation_value', ascending=False)

print("Platform Performance:")
print(platform_perf.to_string())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

sns.barplot(data=social, x='platform', y='estimated_donation_value_php', ax=axes[0,0], palette='viridis')
axes[0,0].set_title('Avg Donation Value by Platform')
axes[0,0].tick_params(axis='x', rotation=45)

sns.barplot(data=social, x='post_type', y='estimated_donation_value_php', ax=axes[0,1], palette='viridis')
axes[0,1].set_title('Avg Donation Value by Post Type')
axes[0,1].tick_params(axis='x', rotation=45)

sns.barplot(data=social, x='has_call_to_action', y='estimated_donation_value_php', ax=axes[0,2], palette='Set2')
axes[0,2].set_title('CTA vs No CTA')

sns.barplot(data=social, x='features_resident_story', y='estimated_donation_value_php', ax=axes[1,0], palette='Set2')
axes[1,0].set_title('Resident Story vs No Story')

sns.barplot(data=social, x='is_boosted', y='estimated_donation_value_php', ax=axes[1,1], palette='Set2')
axes[1,1].set_title('Boosted vs Organic')

sns.barplot(data=social, x='content_topic', y='estimated_donation_value_php', ax=axes[1,2], palette='viridis')
axes[1,2].set_title('Avg Donation Value by Topic')
axes[1,2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Timing analysis
heat = social.pivot_table(index='day_of_week', columns='post_hour', 
                          values='estimated_donation_value_php', aggfunc='mean')
ordered_days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heat = heat.reindex(ordered_days)

plt.figure(figsize=(16, 5))
sns.heatmap(heat, cmap='YlOrRd', linewidths=0.5, annot=False)
plt.title('Average Donation Value by Day of Week and Hour')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.tight_layout()
plt.show()

In [ ]:
# Actual linked donations (ground truth attribution)
donations['donation_date'] = pd.to_datetime(donations['donation_date'])
actual_social = donations[donations['referral_post_id'].notna()].copy()
actual_social['referral_post_id'] = actual_social['referral_post_id'].astype(int)

actual_linked = actual_social.merge(
    social[['post_id', 'platform', 'post_type', 'content_topic', 'has_call_to_action', 
            'features_resident_story', 'is_boosted']],
    left_on='referral_post_id', right_on='post_id', how='inner'
)

print(f"Donations with social media attribution: {len(actual_linked)}")
print(f"\nLinked donations by platform:")
print(actual_linked['platform'].value_counts())
print(f"\nLinked donations by post type:")
print(actual_linked['post_type'].value_counts())

In [ ]:
# Correlation matrix
numeric_cols = ['estimated_donation_value_php', 'engagement_rate', 'click_throughs', 
                'donation_referrals', 'reach', 'impressions', 'likes', 'comments', 'shares',
                'caption_length', 'num_hashtags', 'is_cta', 'is_story', 'is_boosted_flag',
                'follower_count_at_post', 'is_weekend', 'post_hour']
corr = social[numeric_cols].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, 
            annot_kws={'size': 8})
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Modeling & Feature Selection

We build an OLS regression for explanation and tree-based models for prediction. We use `log1p(estimated_donation_value_php)` as the target for regression models to handle the right-skewed distribution, then exponentiate predictions back to PHP.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import statsmodels.api as sm
import joblib

# Define features
numeric_features = ['caption_length', 'num_hashtags', 'mentions_count', 'is_cta', 
                    'is_story', 'is_boosted_flag', 'follower_count_at_post',
                    'is_weekend', 'post_hour']
categorical_features = ['platform', 'post_type', 'media_type', 'content_topic', 
                        'sentiment_tone', 'time_bucket']

# Target: log-transformed donation value
target = 'log_donation_value'

# Drop rows with missing values in features
model_data = social.dropna(subset=numeric_features + categorical_features + [target]).copy()

X = model_data[numeric_features + categorical_features]
y = model_data[target]

# Preprocessing
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

### Explanatory Model: OLS Regression

We use statsmodels OLS to get full coefficient tables with p-values, confidence intervals, and R² — the tools needed for explanation.

In [ ]:
# Prepare data for statsmodels OLS
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = numeric_features + list(
    preprocessor.transformers_[1][1].get_feature_names_out(categorical_features))

X_train_ols = pd.DataFrame(X_train_processed, columns=feature_names)
X_train_ols = sm.add_constant(X_train_ols)

X_test_ols = pd.DataFrame(X_test_processed, columns=feature_names)
X_test_ols = sm.add_constant(X_test_ols)

# Fit OLS
ols_model = sm.OLS(y_train.values, X_train_ols).fit()
print(ols_model.summary())

In [ ]:
# VIF check for multicollinearity in OLS explanatory model
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Use the processed features (without the constant for VIF calculation)
vif_data = pd.DataFrame({
    'Feature': feature_names,
    'VIF': [variance_inflation_factor(X_train_ols.values, i+1) for i in range(len(feature_names))]
}).sort_values('VIF', ascending=False)

print('Variance Inflation Factors (VIF > 10 = problematic multicollinearity):')
print(vif_data.head(15).to_string(index=False))

high_vif = vif_data[vif_data['VIF'] > 10]
if len(high_vif) > 0:
    print(f'\n{len(high_vif)} features with VIF > 10.')
    print('For causal interpretation, these coefficients should be interpreted cautiously:')
    for _, row in high_vif.iterrows():
        print(f'  {row["Feature"]}: VIF = {row["VIF"]:.1f}')
else:
    print('\nNo severe multicollinearity detected.')


In [ ]:
# Visualize significant coefficients
coef_df = pd.DataFrame({
    'feature': ols_model.params.index,
    'coefficient': ols_model.params.values,
    'p_value': ols_model.pvalues.values
})
coef_df = coef_df[coef_df['feature'] != 'const']

# Show features sorted by absolute coefficient
significant = coef_df[coef_df['p_value'] < 0.1].sort_values('coefficient')
not_sig = coef_df[coef_df['p_value'] >= 0.1]

print(f"Significant features (p < 0.1): {len(significant)}")
print(f"Not significant: {len(not_sig)}")

fig, ax = plt.subplots(figsize=(10, max(8, len(significant) * 0.35)))
colors = ['#2ca02c' if c > 0 else '#d62728' for c in significant['coefficient']]
ax.barh(significant['feature'], significant['coefficient'], color=colors)
ax.set_xlabel('OLS Coefficient (log-scale donation value)')
ax.set_title('Significant Drivers of Social Media Donation Conversion')
ax.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

### Predictive Models

In [ ]:
# Build multiple regression models
models = {
    'OLS': Pipeline([('prep', preprocessor), ('reg', LinearRegression())]),
    'Ridge': Pipeline([('prep', preprocessor), ('reg', Ridge(alpha=1.0))]),
    'Decision Tree': Pipeline([('prep', preprocessor), ('reg', DecisionTreeRegressor(random_state=42, max_depth=5))]),
    'Random Forest': Pipeline([('prep', preprocessor), ('reg', RandomForestRegressor(random_state=42, n_estimators=100, max_depth=6))]),
    'Gradient Boosting': Pipeline([('prep', preprocessor), ('reg', GradientBoostingRegressor(random_state=42, n_estimators=100, max_depth=4, learning_rate=0.1))]),
}

results = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    # Convert back from log scale for interpretable metrics
    y_test_php = np.expm1(y_test)
    y_pred_php = np.expm1(y_pred)
    
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='r2')
    
    results[name] = {
        'r2': r2_score(y_test, y_pred),
        'rmse_php': np.sqrt(mean_squared_error(y_test_php, y_pred_php)),
        'mae_php': mean_absolute_error(y_test_php, y_pred_php),
        'cv_r2_mean': cv_scores.mean(),
        'cv_r2_std': cv_scores.std()
    }
    
    print(f"\n{name}")
    print(f"  CV R²: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")
    print(f"  Test R²: {r2_score(y_test, y_pred):.3f}")
    print(f"  RMSE (PHP): {results[name]['rmse_php']:,.0f}")
    print(f"  MAE (PHP):  {results[name]['mae_php']:,.0f}")

In [ ]:
# Hyperparameter tuning for Gradient Boosting
param_grid = {
    'reg__n_estimators': [100, 200, 300],
    'reg__max_depth': [3, 4, 5],
    'reg__learning_rate': [0.05, 0.1, 0.15]
}

gb_pipe = Pipeline([('prep', preprocessor), 
    ('reg', GradientBoostingRegressor(random_state=42))])

grid_search = GridSearchCV(gb_pipe, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV R²: {grid_search.best_score_:.3f}")

best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
print(f"Test R²: {r2_score(y_test, y_pred_best):.3f}")
print(f"Test RMSE (PHP): {np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(y_pred_best))):,.0f}")

In [ ]:
# Feature importance from best model
feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': best_model.named_steps['reg'].feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=feat_imp.head(15), x='importance', y='feature', ax=ax, palette='viridis')
ax.set_title('Feature Importance: Social Media Donation Prediction (Gradient Boosting)')
plt.tight_layout()
plt.show()

## 4. Evaluation & Interpretation

### Metrics
We evaluate using R² (explained variance), RMSE in PHP (interpretable scale), and MAE in PHP. We use cross-validation to ensure the model generalizes.

### Business Interpretation
- **RMSE in PHP** tells the organization how far off their prediction will typically be. If RMSE is 20,000 PHP, the model's predictions are within ~20,000 PHP of actual donation value on average.
- **R²** tells what fraction of variation in donation outcomes is explained by post characteristics (things the organization can control).
- **The explanatory coefficients** are arguably more valuable than the predictions — they translate directly into content strategy decisions.

### Consequences of Errors
- **Overestimating a post's value**: The organization invests time creating a certain type of content that doesn't convert as well as expected. Cost: wasted effort, but no direct harm.
- **Underestimating a post's value**: The organization neglects a content strategy that would have generated significant donations. Cost: missed revenue opportunity.

Neither error is catastrophic, which means the model can be used as advisory guidance rather than requiring extremely high precision.

In [ ]:
# Actual vs predicted scatter plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(np.expm1(y_test), np.expm1(y_pred_best), alpha=0.5, color='#1f77b4')
max_val = max(np.expm1(y_test).max(), np.expm1(y_pred_best).max())
axes[0].plot([0, max_val], [0, max_val], 'r--', alpha=0.7)
axes[0].set_xlabel('Actual Donation Value (PHP)')
axes[0].set_ylabel('Predicted Donation Value (PHP)')
axes[0].set_title('Actual vs Predicted Donation Value')

# Residuals
residuals = np.expm1(y_test) - np.expm1(y_pred_best)
axes[1].hist(residuals, bins=30, color='#2ca02c', edgecolor='white')
axes[1].set_xlabel('Residual (PHP)')
axes[1].set_title('Residual Distribution')
axes[1].axvline(x=0, color='red', linestyle='--')

plt.tight_layout()
plt.show()

# Model comparison table
comparison = pd.DataFrame(results).T.round(3)
print("Model Comparison:")
print(comparison.to_string())

In [ ]:
# Bonus: Classification view — did the post generate ANY donation?
# This confusion matrix helps evaluate the model as a binary classifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# Binary: did this post generate a donation or not?
y_test_binary = (np.expm1(y_test) > 0).astype(int)
y_pred_binary = (np.expm1(y_pred_best) > 0).astype(int)

print('Classification View: Did the post generate any donation?')
print(classification_report(y_test_binary, y_pred_binary, target_names=['No Donation', 'Donation']))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test_binary, y_pred_binary,
    display_labels=['No Donation', 'Donation'], cmap='Blues', ax=ax)
ax.set_title('Confusion Matrix: Post Generated Any Donation?')
plt.tight_layout()
plt.show()


## 5. Causal and Relationship Analysis

### Key Findings from the Explanatory Model

1. **Calls to action drive donations**: Posts with explicit CTAs (DonateNow, LearnMore, etc.) generate significantly more donation value than posts without CTAs. This is the single most actionable finding — every post should include a clear call to action.

2. **Resident stories convert**: Posts featuring anonymized resident stories generate substantially more donations than generic content. This confirms that donors respond to personal narratives — they want to see the human impact of their giving.

3. **Platform matters**: Not all platforms are equal for donation conversion. The model reveals which platforms generate the most donation value per post, which should inform where the organization concentrates its limited posting time.

4. **Boosting has a return**: Paid promotion (boosting) is associated with higher donation value, though the effect must be weighed against the cost. The organization should track ROI on boost spending.

5. **Content topic effects**: Certain topics (e.g., DonorImpact, Reintegration stories) may convert better than others. The organization should prioritize high-converting topics in their content calendar.

6. **Timing effects**: The day of week and hour of posting have measurable effects on donation outcomes, though these may be smaller than content quality factors.

### Causal Defensibility
- **CTA → donations**: This is plausibly causal — providing a clear path to donate removes friction and increases conversion. However, CTAs may also correlate with more carefully crafted posts overall (selection effect).
- **Resident stories → donations**: Likely reflects genuine emotional response to personal narratives, but story posts may also receive more effort in creation. The practical recommendation (use stories) holds regardless.
- **Boosting → donations**: Paid promotion mechanically increases reach, which should increase donations. This is one of the more defensible causal claims, but the key question is ROI, not whether the effect exists.
- **Platform effects**: These reflect both audience composition and platform algorithms. The organization can't change the platform, but they can choose where to focus.
- **We acknowledge that all findings are observational**. The gold standard would be A/B testing different content strategies, which the organization could implement using these findings as hypotheses.

### Content Strategy Recommendations
1. **Always include a CTA** — ideally "DonateNow" or "LearnMore" with a direct link.
2. **Feature resident stories regularly** — aim for at least 1 in 4 posts to include an anonymized impact narrative.
3. **Focus posting effort on the highest-converting platforms** rather than trying to be everywhere.
4. **Test boosting on high-quality posts** and track the donation ROI, not just reach.
5. **Post during the time windows with highest historical conversion** (see timing heatmap).
6. **Track actual donation attribution** (referral_post_id) to build a feedback loop that improves the model over time.

## 6. Deployment Notes

### How This Model Is Deployed
The trained Gradient Boosting model is serialized and served through a .NET API endpoint. 

### Web App Integration
Two integration points:

1. **Social Media Content Advisor (Interactive Tool)**: On the Reports & Analytics page (or a dedicated Social Media Strategy page), staff can select post characteristics — platform, post type, content topic, CTA type, whether to feature a resident story, whether to boost — and the model returns a predicted donation value. This helps staff make data-informed decisions about what to post next.

2. **Content Strategy Insights Panel**: Display the top findings from the explanatory model as actionable cards:
   - "Posts with CTAs generate Xphp more on average"
   - "Resident stories drive X% higher donation conversion"
   - "Your top-converting platform is [Platform]"
   
### Model Export

In [ ]:
# Export model
joblib.dump(best_model, 'social_media_model.pkl')

model_config = {
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'target': 'log1p(estimated_donation_value_php)',
    'note': 'Exponentiate predictions with np.expm1() to get PHP values'
}
import json
with open('social_media_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)
print("Model and config saved.")

In [ ]:
# Example: Content advisor predictions
# Simulate different content strategies
scenarios = pd.DataFrame([
    {'platform': 'Facebook', 'post_type': 'ImpactStory', 'media_type': 'Photo', 
     'content_topic': 'DonorImpact', 'sentiment_tone': 'Hopeful', 'time_bucket': 'Morning',
     'caption_length': 200, 'num_hashtags': 5, 'mentions_count': 2, 
     'is_cta': 1, 'is_story': 1, 'is_boosted_flag': 1, 'follower_count_at_post': 5000,
     'is_weekend': 0, 'post_hour': 10},
    {'platform': 'Instagram', 'post_type': 'FundraisingAppeal', 'media_type': 'Reel', 
     'content_topic': 'Reintegration', 'sentiment_tone': 'Emotional', 'time_bucket': 'Evening',
     'caption_length': 150, 'num_hashtags': 8, 'mentions_count': 1, 
     'is_cta': 1, 'is_story': 1, 'is_boosted_flag': 0, 'follower_count_at_post': 3000,
     'is_weekend': 1, 'post_hour': 19},
    {'platform': 'Twitter', 'post_type': 'EducationalContent', 'media_type': 'Text', 
     'content_topic': 'AwarenessRaising', 'sentiment_tone': 'Informative', 'time_bucket': 'Afternoon',
     'caption_length': 100, 'num_hashtags': 3, 'mentions_count': 0, 
     'is_cta': 0, 'is_story': 0, 'is_boosted_flag': 0, 'follower_count_at_post': 2000,
     'is_weekend': 0, 'post_hour': 14},
])

predictions_log = best_model.predict(scenarios)
predictions_php = np.expm1(predictions_log)

scenarios['predicted_donation_php'] = predictions_php.round(0)
print("Content Strategy Predictions:")
print("=" * 80)
for i, row in scenarios.iterrows():
    print(f"\nScenario {i+1}: {row['platform']} | {row['post_type']} | Story={bool(row['is_story'])} | CTA={bool(row['is_cta'])} | Boosted={bool(row['is_boosted_flag'])}")
    print(f"  Predicted donation value: {row['predicted_donation_php']:,.0f} PHP")